In [13]:
from google.colab import files
import pandas as pd
import string
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [14]:
upload = files.upload()
train_df = pd.read_csv("train.csv")
train_df.head()

Saving train.csv to train (1).csv


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


Question 1 - Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?

In [15]:
freq = train_df['answer'].value_counts()
result = freq.max() + freq.min()

print(freq)
print("\nSum:", result)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

Sum: 814


Question 2 - After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?

In [16]:
def clean_text(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text

cleaned = train_df['prompt'].apply(clean_text)
vocab = set()

for text in cleaned:
    words = text.split()
    vocab.update(words)

len(vocab)

859

Question 3 - Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [17]:
text = train_df.loc[train_df['id'] == 1, 'prompt'].values[0]
text = clean_text(text)

words = text.split()
filtered_words = [w for w in words if w not in ENGLISH_STOP_WORDS]

len(filtered_words)

13

Question 4 - Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [18]:
text_data = (
    train_df['prompt'].fillna('') + ' ' +
    train_df['A'].fillna('') + ' ' +
    train_df['B'].fillna('') + ' ' +
    train_df['C'].fillna('') + ' ' +
    train_df['D'].fillna('') + ' ' +
    train_df['E'].fillna('')
)

corpus = text_data.tolist()
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(corpus)

len(vectorizer.get_feature_names_out())

2762

Question 5 - Using the TF-IDF vectorizer fitted in Question 4, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [19]:
row = train_df.loc[train_df['id'] == 1].iloc[0]
prompt_vec = vectorizer.transform([row['prompt']])
optionA_vec = vectorizer.transform([row['A']])

sim = cosine_similarity(prompt_vec, optionA_vec)[0][0]
sim

np.float64(0.27202429519891635)

Question 6 - Expand the logic from Question 5: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.

In [20]:
prompt_vecs = vectorizer.transform(train_df['prompt'].fillna(''))

option_vecs = {
    opt: vectorizer.transform(train_df[opt].fillna(''))
    for opt in ['A','B','C','D','E']
}

correct = 0

for i in range(len(train_df)):
    sims = {
        opt: cosine_similarity(prompt_vecs[i], option_vecs[opt][i])[0][0]
        for opt in ['A','B','C','D','E']
    }
    predicted = max(sims, key=sims.get)
    if predicted == train_df.iloc[i]['answer']:
        correct += 1

accuracy = (correct / len(train_df)) * 100
accuracy

13.55

Question 7 - If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?

In [21]:
def map_at_3(true_answer, predictions):
    for i, pred in enumerate(predictions[:3], start=1):
        if pred == true_answer:
            return 1 / i
    return 0

map_at_3('C', ['C','A','B'])

1.0

Question 8 - If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?

In [22]:
map_at_3('B', ['D','B','E'])

0.5

Question 9 - The Majority Class Baseline: Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [23]:
freq = train_df['answer'].value_counts()
top3 = freq.index[:3].tolist()

scores = [map_at_3(ans, top3) for ans in train_df['answer']]
map3 = sum(scores) / len(scores)
map3

0.42125

Question 10 - The TF-IDF Pipeline: Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [24]:
scores = []

for i in range(len(train_df)):
    sims = {
        opt: cosine_similarity(prompt_vecs[i], option_vecs[opt][i])[0][0]
        for opt in ['A', 'B', 'C', 'D', 'E']
    }
    ranked = sorted(sims, key=sims.get, reverse=True)
    top3 = ranked[:3]
    score = map_at_3(train_df.iloc[i]['answer'], top3)
    scores.append(score)

final_map3 = sum(scores) / len(scores)
final_map3

0.2961666666666667